# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Metadata is an object, so we just print its properties
print(f"Dataset loaded from: {croissant_url}\n")
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Number of record sets: {len(dataset.record_sets)}")

## 2. Data Overview
Review available record sets, fields, columns and their `@id` references.

In [ ]:
# List available record sets with their @id and name
print("All available record sets in the dataset:")
for rset in dataset.record_sets:
    print(f"- @id: {rset.id} | name: {rset.name}")

# Pick the first record set for further exploration (update if needed)
main_record_set = dataset.record_sets[0]
print(f"\nMain record set chosen: {main_record_set.id} ({main_record_set.name})\n")

print("Fields in the main record set (with @id and name):")
for f in main_record_set.fields:
    print(f"- @id: {f.id} | name: {f.name} | dataType: {f.data_type}")
    if f.column:
        print(f"    column @id: {f.column.id}")

# Print an example record's keys
example_row = next(dataset.records(record_set=main_record_set.id))
print(f"\nExample record keys (field @ids): {list(example_row.keys())}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis.

We use the `@id` of the chosen record set and fields.

In [ ]:
# Collect all record set @ids (show list for reference)
record_set_ids = [rset.id for rset in dataset.record_sets]
print("All record set @ids:", record_set_ids)

# Read all records for each record set as DataFrames (here we load only the main one for demo)
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

# Use main_record_set.id going forward
df = dataframes[main_record_set.id]
print(f"\nColumns (using field @ids): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps using the field `@id`, such as filtering by a threshold, normalization, and grouping. We'll pick a numeric column using its field `@id`.

> If unsure of which field is numeric, use the earlier overview (cell 5) printout. We'll proceed with the first column (or adjust as necessary).

In [ ]:
# Find a numeric field for demo (choose the first that is int/float)
numeric_field_id = None
for f in main_record_set.fields:
    if f.data_type and ("Integer" in f.data_type or "Float" in f.data_type or "Number" in f.data_type):
        numeric_field_id = f.id
        break

if numeric_field_id is None:
    raise Exception("No numeric field found in this record set.")
print(f"Using numeric field @id for demo: {numeric_field_id}")

# Ensure the column is numeric for pandas
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: show only rows with value > threshold
threshold = df[numeric_field_id].quantile(0.5)  # median as threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (median):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
col_norm = f"{numeric_field_id}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, col_norm]].head())

# Choose a grouping field (try to pick a categorical/text one)
group_field_id = None
for f in main_record_set.fields:
    if f.data_type and ("Text" in f.data_type) and f.id != numeric_field_id:
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping in EDA.")

## 5. Visualization

Visualize the numeric field and its distribution, as well as group-wise statistics if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping was possible, show group-wise means
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df, estimator=lambda x: pd.to_numeric(x, errors='coerce').mean(), ci=None)
    plt.title(f"Group-wise mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we explored and processed the FAIR^2 dataset of cancer survivors with second primary colorectal cancer using the `mlcroissant` library. 

- We reviewed dataset metadata and structure using Croissant record set and field `@id`s.
- Data extraction was performed directly using `@id` referencing.
- Numeric fields were filtered and normalized for EDA, with additional breakdown by a text/group field.
- Visualizations summarized distributions and group-wise statistics.

This approach enables reproducible and FAIR-aligned data analysis pipelines. For further modeling, use the same `@id`-based referencing for reproducibility.